In [ ]:
"""Advanced LangGraph Concepts Explained

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1Fh53z-z7F3jd-ibKeaZLnQ4oppXXc-6w
"""

Advanced LangGraph Concepts Demonstration
This script explains complex LangGraph features through a simulated customer support workflow.
We will cover: State Management, State Reducers, Multiple Schemas, Memory/Message Filtering,
and Human-in-the-Loop (HITL).

In [ ]:
import uuid
from typing import TypedDict, List, Annotated, Union
from operator import add

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

--- 1. State Management & Multiple Schemas ---
We'll define two different state schemas (TypedDicts) for different parts of our graph.
This demonstrates how you can manage varied and complex states.

In [ ]:
class UserInfo(TypedDict):
    """Represents basic user information."""
    user_id: str
    name: str
    tier: str # e.g., "free", "premium"

In [ ]:
class Message(TypedDict):
    """Represents a single message in the chat history."""
    sender: str # "user" or "system"
    content: str
    filtered: bool # A flag to show if a message has been filtered

In [ ]:
class MainGraphState(TypedDict):
    """
    This is our primary state schema. It manages the core workflow.
    - messages: The list of messages. We use `add` as a reducer, so new messages are appended.
    - user_info: Static information about the user.
    - external_memory_snapshot: A snapshot of state saved to an "external" system.
    - needs_review: A flag for routing to the Human-in-the-Loop step.
    """
    messages: Annotated[List[Message], add]
    user_info: UserInfo
    external_memory_snapshot: str
    needs_review: bool

In [ ]:
class ReviewSchema(TypedDict):
    """
    A separate, smaller schema for the HITL process.
    This keeps the review process clean and focused.
    """
    review_id: str
    messages_for_review: List[Message]
    decision: str # "approved", "rejected", "pending"

In [ ]:
# --- Mock External Memory (Simulating a Database) ---
# In a real application, this would be a database like Firestore, Redis, or a file system.
external_memory_db = {}

--- Graph Nodes (Workflow Steps) ---

In [ ]:
def start_session(state: MainGraphState) -> MainGraphState:
    """Node: Initializes the session with user data."""
    print("\n--- Node: start_session ---")
    user_info = state['user_info']
    print(f"Initializing session for {user_info['name']} (ID: {user_info['user_id']}).")
    initial_message = Message(
        sender="system",
        content=f"Welcome, {user_info['name']}! Your support session has started.",
        filtered=False
    )
    return {"messages": [initial_message]}

In [ ]:
def message_filter(state: MainGraphState) -> MainGraphState:
    """
    Node: Demonstrates a message filter.
    This node inspects messages and marks certain system messages as "filtered".
    This is useful for creating a cleaner, user-facing chat history.
    """
    print("\n--- Node: message_filter ---")
    print("Applying message filter...")
    updated_messages = []
    for msg in state['messages']:
        # Let's filter out any message that contains the word "internal"
        if "internal" in msg['content'].lower():
            updated_msg = msg.copy()
            updated_msg['filtered'] = True
            updated_messages.append(updated_msg)
            print(f"  - Filtering message: '{msg['content']}'")
        else:
            updated_messages.append(msg)
    # Note: We are replacing the entire list here, not using the reducer.
    return {"messages": updated_messages}

In [ ]:
def memory_trimmer(state: MainGraphState) -> MainGraphState:
    """
    Node: Demonstrates trimming conversation memory.
    To manage context length, this node keeps only the last 4 messages.
    """
    print("\n--- Node: memory_trimmer ---")
    print("Checking memory length...")
    messages = state['messages']
    if len(messages) > 4:
        print(f"  - Trimming memory from {len(messages)} to 4 messages.")
        trimmed_messages = messages[-4:]
        return {"messages": trimmed_messages}
    print("  - Memory length is within limits.")
    return {}

In [ ]:
def save_to_external_memory(state: MainGraphState) -> MainGraphState:
    """
    Node: Simulates saving state to an external source.
    This demonstrates the difference between the graph's internal, volatile memory
    and a persistent, external storage system.
    """
    print("\n--- Node: save_to_external_memory ---")
    user_id = state['user_info']['user_id']
    snapshot = f"Snapshot of {len(state['messages'])} messages for user {user_id}."
    external_memory_db[user_id] = state['messages']
    print(f"  - Saved {len(state['messages'])} messages to external DB for user {user_id}.")
    return {"external_memory_snapshot": snapshot}

In [ ]:
def route_for_review(state: MainGraphState) -> str:
    """Conditional Edge: Decides if human review is needed."""
    print("\n--- Conditional Edge: route_for_review ---")
    # Rule: If a "premium" tier user sends a message containing "urgent", it needs review.
    user_info = state['user_info']
    last_message = state['messages'][-1]
    if user_info['tier'] == "premium" and "urgent" in last_message['content'].lower():
        print("  - Routing to: Human Review (Premium/Urgent)")
        return "request_human_review"
    print("  - Routing to: End of Flow")
    return "end"

In [ ]:
def request_human_review(state: MainGraphState) -> MainGraphState:
    """

    Node: Prepares the state for the human-in-the-loop interruption.
    It flags that a review is needed, which will cause the graph to pause.
    It populates the state with data needed for the separate `ReviewSchema`.
    """
    print("\n--- Node: request_human_review (HITL Entry) ---")
    print("  - Pausing graph execution and waiting for human input.")

    # We are not actually returning a new state for the main graph here.
    # The interruption happens because this node is registered as an interruption point.
    # The data for the review tool is implicitly available in the state.
    return {"needs_review": True}

In [ ]:
# --- Graph Definition ---
def build_graph():
    """Builds and configures the StateGraph."""
    graph_builder = StateGraph(MainGraphState)

    # Add nodes
    graph_builder.add_node("start_session", start_session)
    graph_builder.add_node("message_filter", message_filter)
    graph_builder.add_node("memory_trimmer", memory_trimmer)
    graph_builder.add_node("save_to_external_memory", save_to_external_memory)
    graph_builder.add_node("request_human_review", request_human_review)

    # Define workflow edges
    graph_builder.set_entry_point("start_session")
    graph_builder.add_edge("start_session", "message_filter")
    graph_builder.add_edge("message_filter", "memory_trimmer")
    graph_builder.add_edge("memory_trimmer", "save_to_external_memory")

    # Add conditional edge for HITL routing
    graph_builder.add_conditional_edges(
        "save_to_external_memory",
        route_for_review,
        {
            "request_human_review": "request_human_review",
            "end": END,
        },
    )
    graph_builder.add_edge("request_human_review", END)

    # The `interrupt_before` argument pauses the graph *before* executing the specified nodes.
    return graph_builder.compile(
        checkpointer=MemorySaver(),
        interrupt_before=["request_human_review"]
    )

In [ ]:
# --- Human-in-the-Loop (HITL) Simulation ---
def run_human_review_process(graph_state: MainGraphState):
    """
    This function simulates the human review process. It uses the `ReviewSchema`
    to process the data cleanly.
    """
    print("\n L--- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---")
    print("|   HUMAN-IN-THE-LOOP (HITL) INTERFACE   ")
    print(" L--- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---")

    # Map data from the main graph state to our clean ReviewSchema
    review_data = ReviewSchema(
        review_id=str(uuid.uuid4()),
        messages_for_review=[msg for msg in graph_state['messages'] if not msg['filtered']],
        decision="pending"
    )

    print(f"\nReview Request ID: {review_data['review_id']}")
    print("Messages Requiring Review:")
    for msg in review_data['messages_for_review']:
        print(f"  - [{msg['sender'].capitalize()}]: {msg['content']}")

    # Simulate human input
    decision = ""
    while decision not in ["approve", "reject"]:
        decision = input("\nEnter decision (approve/reject): ").lower().strip()

    review_data['decision'] = "approved" if decision == "approve" else "rejected"
    print(f"\nDecision logged: {review_data['decision'].upper()}.")
    print("Resuming graph execution...")
    print(" L--- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- \n")

In [ ]:
# --- Main Execution ---
if __name__ == "__main__":
    app = build_graph()

    # --- SCENARIO 1: Standard Flow (No HITL) ---
    print("=" * 50)
    print("  RUNNING SCENARIO 1: Standard Flow (No HITL)")
    print("=" * 50)
    config = {"configurable": {"thread_id": "thread-1"}}
    initial_state_1 = {
        "user_info": UserInfo(user_id="user-abc", name="Alice", tier="free"),
        "messages": [
            Message(sender="user", content="Hi, I have a question about my account."),
            Message(sender="system", content="[internal] User is on free tier."),
            Message(sender="user", content="How can I upgrade?"),
        ]
    }
    # Stream events to see the flow
    for event in app.stream(initial_state_1, config):
        # The stream method yields the state *after* each node has executed.
        pass # We're just letting the print statements in the nodes do the work.


    # --- SCENARIO 2: Flow with HITL and Memory Trimming ---
    print("\n" + "=" * 50)
    print("  RUNNING SCENARIO 2: Premium User with HITL")
    print("=" * 50)
    config_2 = {"configurable": {"thread_id": "thread-2"}}
    initial_state_2 = {
        "user_info": UserInfo(user_id="user-xyz", name="Bob", tier="premium"),
        "messages": [
            Message(sender="user", content="My first message."),
            Message(sender="system", content="[internal] Checking user tier."),
            Message(sender="user", content="My second message."),
            Message(sender="user", content="Third message, everything is fine."),
            Message(sender="user", content="This is an URGENT issue with my premium service!"),
        ]
    }
    # Execute the graph until it interrupts
    interrupted_state = None
    for event in app.stream(initial_state_2, config_2):
        if event.get("interrupt"):
            # The 'interrupt' key contains the config needed to resume
            print("\n>>> Graph execution INTERRUPTED for human review. <<<")
            interrupted_config = event["interrupt"]
            # Get the latest state before interruption
            interrupted_state = app.get_state(interrupted_config)
            break

    # If the graph was interrupted, run the human review process
    if interrupted_state and interrupted_state.values['needs_review']:
        run_human_review_process(interrupted_state.values)

        # Resume the graph execution from where it left off
        print("--- Resuming graph from interrupted state... ---")
        for event in app.stream(None, interrupted_config):
             pass # Continue until the end

    print("\n" + "=" * 50)
    print("  DEMONSTRATION COMPLETE")
    print("=" * 50)

    # You can inspect the final states
    final_state_1 = app.get_state(config)
    final_state_2 = app.get_state(config_2)

    print("\nFinal State for Scenario 1 (Alice):")
    print(final_state_1.values)
    print("\nExternal Memory DB content for Alice:")
    print(external_memory_db.get(final_state_1.values['user_info']['user_id']))


    print("\nFinal State for Scenario 2 (Bob):")
    print(final_state_2.values)
    print("\nExternal Memory DB content for Bob (after trimming):")
    print(external_memory_db.get(final_state_2.values['user_info']['user_id']))

Advanced LangGraph Example: A Research Assistant Agent with Tool Calling and Error Handling

This script demonstrates a more complex agent that can:
1.  Create a multi-step plan to answer a research query.
2.  Call different "tools" (simulated functions) to execute the plan.
3.  Handle tool execution errors gracefully.
4.  Use conditional routing to loop through the plan and decide when to finish.
5.  Incorporate Human-in-the-Loop (HITL) for approving costly or sensitive steps.
6.  Manage a more complex state with multiple dynamic fields.

In [ ]:
import uuid
import random
from typing import TypedDict, List, Annotated, Union, Literal
from operator import add

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

--- 1. State Definition ---
This state is more complex. It tracks the plan, tool results, errors,
and the overall conversation history.

In [ ]:
class ToolCall(TypedDict):
    """A structured representation of a tool call in the plan."""
    tool_name: str
    args: dict
    status: Literal["pending", "success", "error"]
    result: Union[str, None]

In [ ]:
class ResearchState(TypedDict):
    """The central state for our research agent."""
    initial_query: str
    plan: List[ToolCall]
    messages: Annotated[List[str], add]
    needs_approval: bool
    final_summary: str

--- 2. Mock Tools ---
These functions simulate external APIs or services the agent can call.

In [ ]:
def web_search(query: str, site_filter: str = None) -> str:
    """Simulates a web search."""
    print(f"  - TOOL(web_search): Searching for '{query}'" + (f" on {site_filter}" if site_filter else ""))
    # Simulate a failure case for demonstration
    if "fail" in query.lower():
        raise ValueError("Web search failed due to network error.")
    return f"Found 3 relevant articles about '{query}'."

In [ ]:
def database_lookup(user_id: str) -> str:
    """Simulates looking up user data."""
    print(f"  - TOOL(database_lookup): Looking up user ID '{user_id}'")
    if user_id == "user-123":
        return "User is a premium subscriber since 2022."
    return "User not found."

In [ ]:
# A dictionary to easily map tool names to their functions.
TOOL_REGISTRY = {
    "web_search": web_search,
    "database_lookup": database_lookup,
}

--- 3. Graph Nodes ---

In [ ]:
def create_plan(state: ResearchState) -> dict:
    """Node: Creates a plan of tool calls to answer the query."""
    print("\n--- Node: create_plan ---")
    query = state['initial_query']
    print(f"Creating a plan for the query: '{query}'")

    # In a real app, an LLM would generate this plan. We'll mock it for predictability.
    plan = [
        ToolCall(tool_name="database_lookup", args={"user_id": "user-123"}, status="pending", result=None),
        ToolCall(tool_name="web_search", args={"query": "latest trends in AI", "site_filter": "techcrunch.com"}, status="pending", result=None),
        ToolCall(tool_name="web_search", args={"query": "impact of AI on finance"}, status="pending", result=None),
    ]
    # Add a failing tool call for demonstration if requested
    if "error" in query.lower():
        plan.append(ToolCall(tool_name="web_search", args={"query": "fail search"}, status="pending", result=None))

    return {"plan": plan, "messages": ["A plan has been created to address your query."]}

In [ ]:
def request_approval(state: ResearchState) -> dict:
    """Node (HITL): Flags that the plan needs human approval before execution."""
    print("\n--- Node: request_approval (HITL Entry) ---")
    print("Plan requires approval. Pausing graph execution.")
    return {"needs_approval": True}

In [ ]:
def execute_next_tool(state: ResearchState) -> dict:
    """Node: Executes the next pending tool call in the plan."""
    print("\n--- Node: execute_next_tool ---")
    plan = state['plan']
    next_tool_call = None
    for i, tool_call in enumerate(plan):
        if tool_call['status'] == 'pending':
            next_tool_call = tool_call
            break

    if not next_tool_call:
        print("All tools have been executed.")
        return {}

    tool_name = next_tool_call['tool_name']
    args = next_tool_call['args']
    print(f"Executing tool: {tool_name} with args: {args}")

    try:
        tool_function = TOOL_REGISTRY[tool_name]
        result = tool_function(**args)
        next_tool_call['status'] = 'success'
        next_tool_call['result'] = result
        message = f"Tool '{tool_name}' executed successfully."
    except Exception as e:
        print(f"  - ERROR: Tool '{tool_name}' failed: {e}")
        next_tool_call['status'] = 'error'
        next_tool_call['result'] = str(e)
        message = f"Error executing tool '{tool_name}': {e}"

    # Update the plan in the state. Must return a copy.
    updated_plan = list(plan)
    return {"plan": updated_plan, "messages": [message]}

In [ ]:
def generate_final_summary(state: ResearchState) -> dict:
    """Node: Generates a final summary based on tool results."""
    print("\n--- Node: generate_final_summary ---")
    results = [tc['result'] for tc in state['plan'] if tc['status'] == 'success']
    # In a real app, an LLM would generate this.
    summary = (
        "Based on the research, here is a summary:\n"
        + "\n".join(f"- {res}" for res in results)
        + "\n\nThis research was conducted for a premium user."
    )
    print("Summary generated.")
    return {"final_summary": summary, "messages": ["Your research summary is complete."]}

--- 4. Conditional Routing ---

In [ ]:
def route_after_planning(state: ResearchState) -> Literal["request_approval", "execute_next_tool"]:
    """Conditional Edge: Decides if the plan needs approval before execution."""
    print("\n--- Conditional Edge: route_after_planning ---")
    # Rule: Any plan with more than 2 steps requires approval.
    if len(state['plan']) > 2:
        print("Routing to: Approval Needed")
        return "request_approval"
    print("Routing to: Execute Tools")
    return "execute_next_tool"

In [ ]:
def route_after_tool_execution(state: ResearchState) -> Literal["generate_final_summary", "execute_next_tool"]:
    """Conditional Edge: Decides the next step after a tool runs."""
    print("\n--- Conditional Edge: route_after_tool_execution ---")
    # If any tools are still pending, loop back to the executor.
    if any(tc['status'] == 'pending' for tc in state['plan']):
        print("Routing to: Execute More Tools (Looping)")
        return "execute_next_tool"
    # Otherwise, proceed to generate the summary.
    print("Routing to: Generate Summary")
    return "generate_final_summary"

--- 5. Graph Definition ---

In [ ]:
def build_graph():
    graph_builder = StateGraph(ResearchState)

    # Add nodes
    graph_builder.add_node("create_plan", create_plan)
    graph_builder.add_node("execute_next_tool", execute_next_tool)
    graph_builder.add_node("generate_final_summary", generate_final_summary)
    graph_builder.add_node("request_approval", request_approval)

    # Set entry point and standard edges
    graph_builder.set_entry_point("create_plan")
    graph_builder.add_edge("request_approval", "execute_next_tool") # After approval, execute
    graph_builder.add_edge("generate_final_summary", END)

    # Add conditional edges for routing
    graph_builder.add_conditional_edges(
        "create_plan",
        route_after_planning,
        {"request_approval": "request_approval", "execute_next_tool": "execute_next_tool"}
    )
    graph_builder.add_conditional_edges(
        "execute_next_tool",
        route_after_tool_execution,
        {"execute_next_tool": "execute_next_tool", "generate_final_summary": "generate_final_summary"}
    )

    return graph_builder.compile(
        checkpointer=MemorySaver(),
        interrupt_before=["request_approval"] # Pause before this node
    )

In [ ]:
# --- 6. Main Execution ---
if __name__ == "__main__":
    app = build_graph()
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    print("=" * 50)
    print("  RUNNING ADVANCED AGENT SCENARIO")
    print("=" * 50)

    # This query will trigger the HITL approval flow
    initial_state = {"initial_query": "Tell me about AI trends for our premium user user-123"}

    # Use .stream() to execute the graph and handle interruptions
    interrupted_config = None
    for event in app.stream(initial_state, config, stream_mode="values"):
        # The 'values' stream mode gives us the full state after each step
        state_snapshot = event

        # Check if the graph has been interrupted
        if app.get_state(config).next:
            if app.get_state(config).next[0] == "request_approval":
                print("\n>>> Graph execution INTERRUPTED for plan approval. <<<")
                interrupted_config = config
                break

    # If the graph was interrupted, simulate the human approval process
    if interrupted_config:
        latest_state = app.get_state(interrupted_config).values
        print("\n L--- HUMAN-IN-THE-LOOP (HITL) INTERFACE ---")
        print("A plan has been generated and requires your approval:")
        for i, tool_call in enumerate(latest_state['plan']):
            print(f"  Step {i+1}: {tool_call['tool_name']}({tool_call['args']})")

        approval = ""
        while approval not in ["yes", "no"]:
            approval = input("Do you approve this plan? (yes/no): ").lower().strip()

        print("--- Resuming graph execution... ---\n")
        if approval == "yes":
            # To resume, we stream with a `None` input and the interrupted config
            for _ in app.stream(None, interrupted_config):
                pass
        else:
            print("Plan rejected. Ending execution.")

    print("\n" + "=" * 50)
    print("  WORKFLOW COMPLETE")
    print("=" * 50)

    # Inspect the final state of the graph
    final_state = app.get_state(config).values
    print("\n--- FINAL STATE ---")
    print(f"Initial Query: {final_state.get('initial_query')}")
    print("\nFinal Plan Execution Status:")
    for tc in final_state.get('plan', []):
        print(f"  - Tool: {tc['tool_name']}, Status: {tc['status']}, Result: {tc['result']}")
    print(f"\nFinal Summary:\n{final_state.get('final_summary')}")

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

In [ ]:
class State(TypedDict):
    messages: list[str]  # Default: overwrite
    count: Annotated[int, add] # Reducer: add updates to count

In [ ]:
!pip install langgraph

In [ ]:
from typing import TypedDict, List

In [ ]:
class State(TypedDict):
    messages: List[dict]

In [ ]:
def memory_trimmer(state: State) -> dict:
    """
    Checks the message history and keeps only the last 10 messages.
    """
    print("\n--- Node: memory_trimmer ---")

    # Define the maximum number of messages to keep
    MAX_MESSAGES = 10

    messages = state['messages']

    if len(messages) > MAX_MESSAGES:
        print(f"  - Trimming memory from {len(messages)} to {MAX_MESSAGES} messages.")

        # This slice keeps the last 10 items in the list
        trimmed_messages = messages[-MAX_MESSAGES:]

        # Return the trimmed list to overwrite the old one in the state
        return {"messages": trimmed_messages}

    print("  - Memory length is within limits. No trimming needed.")
    return {}